<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/DL/Osnovy_NNDZ_Kondratev_Osnovy_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание по теме «Основы нейронных сетей».

**Цель задания:**

научиться обучать простейшую нейросетевую модель, практически эквивалентную алгоритму линейной регрессии, с помощью метода градиентного спуска в pytorch

**Задание**

Реализуйте обучение нейронной сети из одного нейрона для задачи предсказания стоимости квартир boston house prices или california housing prices с использованием pytorch.

**Инструкция к выполнению задания**

Загрузите и подготовьте данные

Разделите данные на test и train

Создать модель (объект) класса. Для создания объекта можно использовать класс Sequential

Обучите модель на train данных

Проверьте качество модели на тестовых данных

## 1. Понимание бизнеса

### Цель проекта
Реализовать обучение простейшей нейронной сети (один нейрон) для задачи предсказания стоимости жилья с использованием PyTorch.

### Проблема бизнеса
Разработка модели, способной предсказывать стоимость жилья на основе различных характеристик недвижимости, что может быть полезно для риелторов, покупателей и продавцов недвижимости.

### Критерии успеха
- Успешная реализация нейронной сети с одним нейроном в PyTorch
- Достижение адекватных метрик качества (MSE, MAE, R²) на тестовых данных
- Сравнение результатов с базовой моделью (например, линейной регрессией)

### Риски и ограничения
- Ограниченная выразительность модели с одним нейроном
- Возможные проблемы с качеством и размером данных
- Нестабильность обучения из-за неоптимальных гиперпараметров

## 2. Понимание данных

### Подключение библиотек

In [1]:
# Базовые библиотеки для анализа данных
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Библиотеки для работы с моделями
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

# Библиотеки для оценки качества моделей
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Библиотеки для загрузки данных
from sklearn.datasets import fetch_california_housing

# Настройки отображения
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
sns.set(font_scale=1.2)


### Сбор данных

In [2]:
# Загрузка набора данных California Housing
housing = fetch_california_housing()
X = housing.data
y = housing.target

# Создание DataFrame для удобства работы
feature_names = housing.feature_names
df = pd.DataFrame(X, columns=feature_names)
df['PRICE'] = y

# Проверка размера данных
print(f"Размер данных: {df.shape}")
print("\nПервые 5 строк данных:")
print(df.head())
print("\nОбщая информация о данных:")
print(df.info())


Размер данных: (20640, 9)

Первые 5 строк данных:
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  PRICE  
0    -122.23  4.526  
1    -122.22  3.585  
2    -122.24  3.521  
3    -122.25  3.413  
4    -122.25  3.422  

Общая информация о данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   HouseAge    20640 non-null  float64
 2   AveRooms    20640 non-null  float64
 3   AveBedrms   206

### Описание данных

In [3]:
# Создание таблицы с описанием переменных
feature_descriptions = {
    'MedInc': 'Медианный доход домохозяйства в блоке',
    'HouseAge': 'Средний возраст домов в блоке',
    'AveRooms': 'Среднее количество комнат на дом',
    'AveBedrms': 'Среднее количество спален на дом',
    'Population': 'Население блока',
    'AveOccup': 'Среднее количество жильцов в доме',
    'Latitude': 'Широта блока',
    'Longitude': 'Долгота блока',
    'PRICE': 'Средняя стоимость дома в блоке (в сотнях тысяч долларов)'
}

features_df = pd.DataFrame({
    'Переменная': feature_names + ['PRICE'],
    'Описание': [feature_descriptions[feature] for feature in feature_names + ['PRICE']]
})

print("Описание переменных:")
print(features_df)


Описание переменных:
   Переменная                                           Описание
0      MedInc              Медианный доход домохозяйства в блоке
1    HouseAge                      Средний возраст домов в блоке
2    AveRooms                   Среднее количество комнат на дом
3   AveBedrms                   Среднее количество спален на дом
4  Population                                    Население блока
5    AveOccup                  Среднее количество жильцов в доме
6    Latitude                                       Широта блока
7   Longitude                                      Долгота блока
8       PRICE  Средняя стоимость дома в блоке (в сотнях тысяч...


### Описательная статистика

In [4]:
# Получение основных статистических показателей
stats = df.describe()
print("Основные статистические показатели:")
print(stats)

# Дополнительные статистики по целевой переменной
target_stats = pd.DataFrame({
    'Метрика': ['Минимум', 'Максимум', 'Среднее', 'Медиана', 'Стандартное отклонение', 'Асимметрия', 'Эксцесс'],
    'Значение': [
        df['PRICE'].min(),
        df['PRICE'].max(),
        df['PRICE'].mean(),
        df['PRICE'].median(),
        df['PRICE'].std(),
        df['PRICE'].skew(),
        df['PRICE'].kurt()
    ]
})

print("\nСтатистика целевой переменной 'PRICE':")
print(target_stats)


Основные статистические показатели:
             MedInc      HouseAge      AveRooms     AveBedrms    Population  \
count  20640.000000  20640.000000  20640.000000  20640.000000  20640.000000   
mean       3.870671     28.639486      5.429000      1.096675   1425.476744   
std        1.899822     12.585558      2.474173      0.473911   1132.462122   
min        0.499900      1.000000      0.846154      0.333333      3.000000   
25%        2.563400     18.000000      4.440716      1.006079    787.000000   
50%        3.534800     29.000000      5.229129      1.048780   1166.000000   
75%        4.743250     37.000000      6.052381      1.099526   1725.000000   
max       15.000100     52.000000    141.909091     34.066667  35682.000000   

           AveOccup      Latitude     Longitude         PRICE  
count  20640.000000  20640.000000  20640.000000  20640.000000  
mean       3.070655     35.631861   -119.569704      2.068558  
std       10.386050      2.135952      2.003532      1.15395

### Исследовательский анализ данных (EDA)

#### Распределения

In [5]:
# Создаем фигуру для гистограмм всех признаков
plt.figure(figsize=(15, 12))

# Строим гистограммы для всех признаков
for i, feature in enumerate(df.columns):
    plt.subplot(3, 3, i+1)
    sns.histplot(df[feature], kde=True)
    plt.title(f'Распределение {feature}')
    plt.tight_layout()

plt.savefig('histograms.png')
plt.close()

# Boxplots для всех признаков
plt.figure(figsize=(15, 12))
for i, feature in enumerate(df.columns):
    plt.subplot(3, 3, i+1)
    sns.boxplot(y=df[feature])
    plt.title(f'Boxplot для {feature}')
    plt.tight_layout()

plt.savefig('boxplots.png')
plt.close()

# KDE plots для всех признаков
plt.figure(figsize=(15, 12))
for i, feature in enumerate(df.columns):
    plt.subplot(3, 3, i+1)
    sns.kdeplot(df[feature], fill=True)
    plt.title(f'KDE для {feature}')
    plt.tight_layout()

plt.savefig('kdeplots.png')
plt.close()

print("Гистограммы, боксплоты и KDE-графики сохранены в файлы histograms.png, boxplots.png и kdeplots.png")


Гистограммы, боксплоты и KDE-графики сохранены в файлы histograms.png, boxplots.png и kdeplots.png


#### Корреляции

In [6]:
# Расчет корреляционной матрицы
corr_matrix = df.corr()

# Построение тепловой карты корреляций
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Тепловая карта корреляций')
plt.tight_layout()
plt.savefig('correlation_heatmap.png')
plt.close()

# Корреляция с целевой переменной
target_corr = corr_matrix['PRICE'].sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=target_corr.index, y=target_corr.values)
plt.title('Корреляция признаков с целевой переменной PRICE')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('target_correlation.png')
plt.close()

print("Тепловая карта корреляций сохранена в файл correlation_heatmap.png")
print("График корреляций с целевой переменной сохранен в файл target_correlation.png")
print("\nКорреляция признаков с целевой переменной PRICE:")
print(target_corr)


Тепловая карта корреляций сохранена в файл correlation_heatmap.png
График корреляций с целевой переменной сохранен в файл target_correlation.png

Корреляция признаков с целевой переменной PRICE:
PRICE         1.000000
MedInc        0.688075
AveRooms      0.151948
HouseAge      0.105623
AveOccup     -0.023737
Population   -0.024650
Longitude    -0.045967
AveBedrms    -0.046701
Latitude     -0.144160
Name: PRICE, dtype: float64


#### Анализ целевой переменной

In [7]:
# Дополнительный анализ целевой переменной
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
sns.histplot(df['PRICE'], kde=True)
plt.title('Распределение цен на жилье')
plt.xlabel('Цена (сотни тысяч долларов)')
plt.ylabel('Частота')

plt.subplot(1, 2, 2)
sns.boxplot(y=df['PRICE'])
plt.title('Boxplot цен на жилье')
plt.ylabel('Цена (сотни тысяч долларов)')

plt.tight_layout()
plt.savefig('target_analysis.png')
plt.close()

# Анализ цен в зависимости от медианного дохода (самая высокая корреляция)
plt.figure(figsize=(10, 6))
sns.scatterplot(x='MedInc', y='PRICE', data=df, alpha=0.5)
plt.title('Зависимость цены от медианного дохода')
plt.xlabel('Медианный доход')
plt.ylabel('Цена (сотни тысяч долларов)')
plt.savefig('price_vs_income.png')
plt.close()

# Анализ цен в зависимости от географического положения
plt.figure(figsize=(12, 8))
scatter = plt.scatter(df['Longitude'], df['Latitude'], c=df['PRICE'],
                      cmap='viridis', alpha=0.5, s=10)
plt.colorbar(scatter, label='Цена (сотни тысяч долларов)')
plt.title('Географическое распределение цен на жилье')
plt.xlabel('Долгота')
plt.ylabel('Широта')
plt.savefig('geo_price_distribution.png')
plt.close()

print("Анализ целевой переменной сохранен в файл target_analysis.png")
print("График зависимости цены от медианного дохода сохранен в файл price_vs_income.png")
print("Географическое распределение цен сохранено в файл geo_price_distribution.png")


Анализ целевой переменной сохранен в файл target_analysis.png
График зависимости цены от медианного дохода сохранен в файл price_vs_income.png
Географическое распределение цен сохранено в файл geo_price_distribution.png


#### Отчет о качестве данных

In [9]:
# Проверка пропущенных значений
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100

# Создание таблицы с пропущенными значениями
missing_data = pd.DataFrame({
    'Количество пропущенных значений': missing_values,
    'Процент пропущенных значений': missing_percentage
})

print("Анализ пропущенных значений:")
print(missing_data)

# Проверка выбросов с использованием 1.5 * IQR
def detect_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return len(outliers), (len(outliers) / len(df)) * 100

# Создание таблицы с выбросами
outliers_data = []
for column in df.columns:
    count, percentage = detect_outliers(df, column)
    outliers_data.append({
        'Признак': column,
        'Количество выбросов': count,
        'Процент выбросов': percentage
    })

outliers_df = pd.DataFrame(outliers_data)
print("\nАнализ выбросов:")
print(outliers_df)

# Проверка дубликатов
duplicates = df.duplicated().sum()
print(f"\nКоличество дубликатов: {duplicates}")
print(f"Процент дубликатов: {(duplicates / len(df)) * 100:.2f}%")

# Проверка на несбалансированность целевой переменной
# Разделим целевую переменную на бины для анализа распределения
bins = np.linspace(df['PRICE'].min(), df['PRICE'].max(), 10)
price_binned = pd.cut(df['PRICE'], bins)
price_distribution = price_binned.value_counts().sort_index()

print("\nРаспределение целевой переменной по бинам:")
print(price_distribution)


Анализ пропущенных значений:
            Количество пропущенных значений  Процент пропущенных значений
MedInc                                    0                           0.0
HouseAge                                  0                           0.0
AveRooms                                  0                           0.0
AveBedrms                                 0                           0.0
Population                                0                           0.0
AveOccup                                  0                           0.0
Latitude                                  0                           0.0
Longitude                                 0                           0.0
PRICE                                     0                           0.0

Анализ выбросов:
      Признак  Количество выбросов  Процент выбросов
0      MedInc                  681          3.299419
1    HouseAge                    0          0.000000
2    AveRooms                  511          2.475775
3

### Подготовка данных

#### Очистка данных

In [10]:
# Поскольку в данных нет пропущенных значений, нам не нужно их обрабатывать
# Но мы можем обработать выбросы для некоторых признаков

# Создадим копию DataFrame для предобработки
df_processed = df.copy()

# Функция для обработки выбросов методом винсоризации
def winsorize_column(df, column, lower_quantile=0.01, upper_quantile=0.99):
    lower_bound = df[column].quantile(lower_quantile)
    upper_bound = df[column].quantile(upper_quantile)
    df[column] = df[column].clip(lower=lower_bound, upper=upper_bound)
    return df

# Обработаем выбросы в признаках с большим процентом выбросов
columns_to_winsorize = ['AveBedrms', 'Population', 'AveOccup', 'AveRooms']
for column in columns_to_winsorize:
    df_processed = winsorize_column(df_processed, column)

# Проверим, как изменились данные после обработки выбросов
print("Статистика до обработки выбросов:")
print(df[columns_to_winsorize].describe())

print("\nСтатистика после обработки выбросов:")
print(df_processed[columns_to_winsorize].describe())

# Проверим количество оставшихся выбросов после обработки
outliers_after = []
for column in columns_to_winsorize:
    count, percentage = detect_outliers(df_processed, column)
    outliers_after.append({
        'Признак': column,
        'Количество выбросов': count,
        'Процент выбросов': percentage
    })

outliers_after_df = pd.DataFrame(outliers_after)
print("\nАнализ выбросов после обработки:")
print(outliers_after_df)

# Выведем первые строки обработанного датасета
print("\nПервые 5 строк обработанного датасета:")
print(df_processed.head())


Статистика до обработки выбросов:
          AveBedrms    Population      AveOccup      AveRooms
count  20640.000000  20640.000000  20640.000000  20640.000000
mean       1.096675   1425.476744      3.070655      5.429000
std        0.473911   1132.462122     10.386050      2.474173
min        0.333333      3.000000      0.692308      0.846154
25%        1.006079    787.000000      2.429741      4.440716
50%        1.048780   1166.000000      2.818116      5.229129
75%        1.099526   1725.000000      3.282261      6.052381
max       34.066667  35682.000000   1243.333333    141.909091

Статистика после обработки выбросов:
          AveBedrms    Population      AveOccup      AveRooms
count  20640.000000  20640.000000  20640.000000  20640.000000
mean       1.077081   1404.011958      2.916367      5.334186
std        0.158544    972.927381      0.732299      1.321643
min        0.872840     88.000000      1.536686      2.581133
25%        1.006079    787.000000      2.429741      4.44071

#### Создание признаков

In [11]:
# Создадим некоторые новые признаки, которые могут быть полезны для модели

# 1. Отношение комнат к спальням
df_processed['RoomBedroomRatio'] = df_processed['AveRooms'] / df_processed['AveBedrms']

# 2. Плотность населения (население / количество домов)
df_processed['PopulationDensity'] = df_processed['Population'] / df_processed['AveOccup']

# 3. Расстояние от центра Калифорнии (примерные координаты)
center_lat, center_lon = 36.7783, -119.4179
df_processed['DistanceFromCenter'] = np.sqrt(
    (df_processed['Latitude'] - center_lat)**2 +
    (df_processed['Longitude'] - center_lon)**2
)

# 4. Квадраты некоторых признаков для учета нелинейных зависимостей
df_processed['MedInc_Squared'] = df_processed['MedInc'] ** 2
df_processed['HouseAge_Squared'] = df_processed['HouseAge'] ** 2

# 5. Взаимодействие признаков
df_processed['Income_Age_Interaction'] = df_processed['MedInc'] * df_processed['HouseAge']

# Проверим корреляцию новых признаков с целевой переменной
new_features = ['RoomBedroomRatio', 'PopulationDensity', 'DistanceFromCenter',
                'MedInc_Squared', 'HouseAge_Squared', 'Income_Age_Interaction']

new_features_corr = df_processed[new_features + ['PRICE']].corr()['PRICE'].sort_values(ascending=False)

print("Корреляция новых признаков с целевой переменной:")
print(new_features_corr)

# Выведем первые строки датасета с новыми признаками
print("\nПервые 5 строк датасета с новыми признаками:")
print(df_processed[new_features].head())


Корреляция новых признаков с целевой переменной:
PRICE                     1.000000
MedInc_Squared            0.624514
Income_Age_Interaction    0.589142
RoomBedroomRatio          0.384523
HouseAge_Squared          0.119955
DistanceFromCenter        0.078628
PopulationDensity         0.070985
Name: PRICE, dtype: float64

Первые 5 строк датасета с новыми признаками:
   RoomBedroomRatio  PopulationDensity  DistanceFromCenter  MedInc_Squared  \
0          6.821705              126.0            3.020207       69.308955   
1          6.418626             1138.0            3.003638       68.913242   
2          7.721053              177.0            3.018740       52.669855   
3          5.421277              219.0            3.028090       31.844578   
4          5.810714              259.0            3.028090       14.793254   

   HouseAge_Squared  Income_Age_Interaction  
0            1681.0                341.3332  
1             441.0                174.3294  
2            2704.0      

#### Разделение данных

In [12]:
# Определим признаки и целевую переменную
X = df_processed.drop('PRICE', axis=1)
y = df_processed['PRICE']

# Разделим данные на обучающую, валидационную и тестовую выборки
# Сначала отделяем тестовую выборку (20%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Затем разделяем оставшиеся данные на обучающую и валидационную выборки (80% и 20% от оставшихся данных)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

# Проверяем размеры полученных выборок
print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер валидационной выборки: {X_val.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

# Стандартизируем данные для лучшего обучения нейронной сети
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Преобразуем данные в тензоры PyTorch
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

X_val_tensor = torch.FloatTensor(X_val_scaled)
y_val_tensor = torch.FloatTensor(y_val.values).reshape(-1, 1)

X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)

# Создаем датасеты и загрузчики данных для PyTorch
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

print(f"\nКоличество батчей в обучающем загрузчике: {len(train_loader)}")
print(f"Количество батчей в валидационном загрузчике: {len(val_loader)}")
print(f"Количество батчей в тестовом загрузчике: {len(test_loader)}")

# Проверим размерность входных данных для модели
input_dim = X_train_scaled.shape[1]
print(f"\nРазмерность входных данных: {input_dim}")


Размер обучающей выборки: (12384, 14)
Размер валидационной выборки: (4128, 14)
Размер тестовой выборки: (4128, 14)

Количество батчей в обучающем загрузчике: 194
Количество батчей в валидационном загрузчике: 65
Количество батчей в тестовом загрузчике: 65

Размерность входных данных: 14


### Моделирование

#### Выбор моделей

In [13]:
# Определим модель нейронной сети с одним нейроном (линейная модель)
class SimpleNeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(SimpleNeuralNetwork, self).__init__()
        self.linear = nn.Linear(input_size, 1)

    def forward(self, x):
        return self.linear(x)

# Создадим также модель с одним скрытым слоем для сравнения
class NeuralNetworkWithHiddenLayer(nn.Module):
    def __init__(self, input_size, hidden_size=8):
        super(NeuralNetworkWithHiddenLayer, self).__init__()
        self.hidden = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.hidden(x)
        x = self.activation(x)
        return self.output(x)

# Инициализируем модели
model_simple = SimpleNeuralNetwork(input_dim)
model_with_hidden = NeuralNetworkWithHiddenLayer(input_dim)

# Определим функцию потерь и оптимизатор для обеих моделей
criterion = nn.MSELoss()
optimizer_simple = optim.Adam(model_simple.parameters(), lr=0.001)
optimizer_with_hidden = optim.Adam(model_with_hidden.parameters(), lr=0.001)

# Выведем архитектуру моделей
print("Архитектура простой модели (один нейрон):")
print(model_simple)

print("\nАрхитектура модели со скрытым слоем:")
print(model_with_hidden)

print("\nСписок выбранных моделей:")
print("1. Простая нейронная сеть с одним нейроном (эквивалент линейной регрессии)")
print("2. Нейронная сеть с одним скрытым слоем (8 нейронов) и ReLU активацией")


Архитектура простой модели (один нейрон):
SimpleNeuralNetwork(
  (linear): Linear(in_features=14, out_features=1, bias=True)
)

Архитектура модели со скрытым слоем:
NeuralNetworkWithHiddenLayer(
  (hidden): Linear(in_features=14, out_features=8, bias=True)
  (activation): ReLU()
  (output): Linear(in_features=8, out_features=1, bias=True)
)

Список выбранных моделей:
1. Простая нейронная сеть с одним нейроном (эквивалент линейной регрессии)
2. Нейронная сеть с одним скрытым слоем (8 нейронов) и ReLU активацией


#### Обучение моделей

In [14]:
# Функция для обучения модели
def train_model(model, optimizer, train_loader, val_loader, num_epochs=100, early_stopping_patience=10):
    # Для отслеживания прогресса обучения
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None

    for epoch in range(num_epochs):
        # Обучение
        model.train()
        train_loss = 0.0
        for inputs, targets in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # Валидация
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in val_loader:
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        # Вывод прогресса каждые 10 эпох
        if (epoch + 1) % 10 == 0:
            print(f'Эпоха [{epoch+1}/{num_epochs}], Потери на обучении: {train_loss:.4f}, Потери на валидации: {val_loss:.4f}')

        # Раннее останавливание
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print(f'Раннее останавливание на эпохе {epoch+1}')
                # Восстановление лучшего состояния модели
                model.load_state_dict(best_model_state)
                break

    return train_losses, val_losses

# Обучение простой модели
print("Обучение простой модели (один нейрон):")
train_losses_simple, val_losses_simple = train_model(model_simple, optimizer_simple, train_loader, val_loader)

# Обучение модели со скрытым слоем
print("\nОбучение модели со скрытым слоем:")
train_losses_hidden, val_losses_hidden = train_model(model_with_hidden, optimizer_with_hidden, train_loader, val_loader)

# Построение графика потерь
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses_simple, label='Train Loss')
plt.plot(val_losses_simple, label='Validation Loss')
plt.title('Потери простой модели')
plt.xlabel('Эпоха')
plt.ylabel('Потери (MSE)')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_losses_hidden, label='Train Loss')
plt.plot(val_losses_hidden, label='Validation Loss')
plt.title('Потери модели со скрытым слоем')
plt.xlabel('Эпоха')
plt.ylabel('Потери (MSE)')
plt.legend()

plt.tight_layout()
plt.savefig('training_losses.png')
plt.close()

# Оценка моделей на валидационном наборе
def evaluate_model(model, data_loader):
    model.eval()
    predictions = []
    actuals = []

    with torch.no_grad():
        for inputs, targets in data_loader:
            outputs = model(inputs)
            predictions.extend(outputs.numpy().flatten())
            actuals.extend(targets.numpy().flatten())

    predictions = np.array(predictions)
    actuals = np.array(actuals)

    mse = mean_squared_error(actuals, predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(actuals, predictions)
    r2 = r2_score(actuals, predictions)

    return {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2
    }

# Оценка моделей на валидационном наборе
val_metrics_simple = evaluate_model(model_simple, val_loader)
val_metrics_hidden = evaluate_model(model_with_hidden, val_loader)

print("\nМетрики на валидационных данных для простой модели:")
for metric, value in val_metrics_simple.items():
    print(f"{metric}: {value:.4f}")

print("\nМетрики на валидационных данных для модели со скрытым слоем:")
for metric, value in val_metrics_hidden.items():
    print(f"{metric}: {value:.4f}")


Обучение простой модели (один нейрон):
Эпоха [10/100], Потери на обучении: 0.8773, Потери на валидации: 0.8214
Эпоха [20/100], Потери на обучении: 0.4518, Потери на валидации: 0.4615
Эпоха [30/100], Потери на обучении: 0.4406, Потери на валидации: 0.4503
Эпоха [40/100], Потери на обучении: 0.4405, Потери на валидации: 0.4488
Эпоха [50/100], Потери на обучении: 0.4393, Потери на валидации: 0.4477
Эпоха [60/100], Потери на обучении: 0.4389, Потери на валидации: 0.4475
Эпоха [70/100], Потери на обучении: 0.4403, Потери на валидации: 0.4479
Эпоха [80/100], Потери на обучении: 0.4390, Потери на валидации: 0.4482
Эпоха [90/100], Потери на обучении: 0.4389, Потери на валидации: 0.4472
Эпоха [100/100], Потери на обучении: 0.4394, Потери на валидации: 0.4470

Обучение модели со скрытым слоем:
Эпоха [10/100], Потери на обучении: 0.3794, Потери на валидации: 0.3896
Эпоха [20/100], Потери на обучении: 0.3407, Потери на валидации: 0.3589
Эпоха [30/100], Потери на обучении: 0.3278, Потери на валидац

#### Оптимизация гиперпараметров

In [15]:
# Определим функцию для обучения модели с заданными гиперпараметрами
def train_model_with_params(model_class, input_dim, learning_rate, hidden_size=None, weight_decay=0):
    # Создаем модель
    if model_class == SimpleNeuralNetwork:
        model = model_class(input_dim)
    else:
        model = model_class(input_dim, hidden_size)

    # Определяем оптимизатор с заданными параметрами
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Обучаем модель
    train_losses, val_losses = train_model(model, optimizer, train_loader, val_loader,
                                           num_epochs=100, early_stopping_patience=10)

    # Оцениваем модель на валидационном наборе
    val_metrics = evaluate_model(model, val_loader)

    return model, val_metrics, val_losses[-1]

# Определим гиперпараметры для оптимизации простой модели
learning_rates_simple = [0.001, 0.005, 0.01]
weight_decays_simple = [0, 0.001, 0.01]

# Оптимизация гиперпараметров для простой модели
best_val_loss_simple = float('inf')
best_params_simple = {}
best_model_simple = None

print("Оптимизация гиперпараметров для простой модели:")
for lr in learning_rates_simple:
    for wd in weight_decays_simple:
        print(f"\nОбучение с learning_rate={lr}, weight_decay={wd}")
        model, metrics, val_loss = train_model_with_params(SimpleNeuralNetwork, input_dim,
                                                          learning_rate=lr, weight_decay=wd)

        print(f"Валидационные метрики: MSE={metrics['MSE']:.4f}, RMSE={metrics['RMSE']:.4f}, R²={metrics['R²']:.4f}")

        if val_loss < best_val_loss_simple:
            best_val_loss_simple = val_loss
            best_params_simple = {'learning_rate': lr, 'weight_decay': wd}
            best_model_simple = model

# Определим гиперпараметры для оптимизации модели со скрытым слоем
learning_rates_hidden = [0.001, 0.005]
hidden_sizes = [8, 16, 32]
weight_decays_hidden = [0, 0.001]

# Оптимизация гиперпараметров для модели со скрытым слоем
best_val_loss_hidden = float('inf')
best_params_hidden = {}
best_model_hidden = None

print("\nОптимизация гиперпараметров для модели со скрытым слоем:")
for lr in learning_rates_hidden:
    for hs in hidden_sizes:
        for wd in weight_decays_hidden:
            print(f"\nОбучение с learning_rate={lr}, hidden_size={hs}, weight_decay={wd}")
            model, metrics, val_loss = train_model_with_params(NeuralNetworkWithHiddenLayer, input_dim,
                                                              learning_rate=lr, hidden_size=hs, weight_decay=wd)

            print(f"Валидационные метрики: MSE={metrics['MSE']:.4f}, RMSE={metrics['RMSE']:.4f}, R²={metrics['R²']:.4f}")

            if val_loss < best_val_loss_hidden:
                best_val_loss_hidden = val_loss
                best_params_hidden = {'learning_rate': lr, 'hidden_size': hs, 'weight_decay': wd}
                best_model_hidden = model

# Вывод лучших параметров
print("\nЛучшие параметры для простой модели:")
for param, value in best_params_simple.items():
    print(f"{param}: {value}")
print(f"Лучшие потери на валидации: {best_val_loss_simple:.4f}")

print("\nЛучшие параметры для модели со скрытым слоем:")
for param, value in best_params_hidden.items():
    print(f"{param}: {value}")
print(f"Лучшие потери на валидации: {best_val_loss_hidden:.4f}")

# Сохраняем лучшие модели
model_simple = best_model_simple
model_with_hidden = best_model_hidden


Оптимизация гиперпараметров для простой модели:

Обучение с learning_rate=0.001, weight_decay=0
Эпоха [10/100], Потери на обучении: 1.0273, Потери на валидации: 0.9571
Эпоха [20/100], Потери на обучении: 0.4645, Потери на валидации: 0.4742
Эпоха [30/100], Потери на обучении: 0.4445, Потери на валидации: 0.4557
Эпоха [40/100], Потери на обучении: 0.4402, Потери на валидации: 0.4511
Эпоха [50/100], Потери на обучении: 0.4396, Потери на валидации: 0.4483
Эпоха [60/100], Потери на обучении: 0.4384, Потери на валидации: 0.4478
Эпоха [70/100], Потери на обучении: 0.4391, Потери на валидации: 0.4479
Эпоха [80/100], Потери на обучении: 0.4390, Потери на валидации: 0.4472
Эпоха [90/100], Потери на обучении: 0.4382, Потери на валидации: 0.4472
Раннее останавливание на эпохе 91
Валидационные метрики: MSE=0.4482, RMSE=0.6694, R²=0.6736

Обучение с learning_rate=0.001, weight_decay=0.001
Эпоха [10/100], Потери на обучении: 0.9666, Потери на валидации: 0.8987
Эпоха [20/100], Потери на обучении: 0.45

#### Важность признаков

In [16]:
# Анализ важности признаков для простой модели
def get_feature_importance(model, feature_names):
    # Для простой модели важность признаков пропорциональна весам
    if isinstance(model, SimpleNeuralNetwork):
        weights = model.linear.weight.data.numpy()[0]
    else:
        # Для модели со скрытым слоем используем произведение весов
        input_weights = model.hidden.weight.data.numpy()
        output_weights = model.output.weight.data.numpy()

        # Вычисляем важность как произведение весов входного и выходного слоя
        weights = np.zeros(len(feature_names))
        for i in range(len(feature_names)):
            for j in range(input_weights.shape[0]):
                weights[i] += abs(input_weights[j, i] * output_weights[0, j])

    # Нормализуем веса для лучшего отображения
    abs_weights = np.abs(weights)
    normalized_weights = abs_weights / np.sum(abs_weights)

    # Создаем DataFrame с важностью признаков
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': normalized_weights,
        'AbsWeight': abs_weights,
        'Weight': weights
    })

    # Сортируем по убыванию важности
    importance_df = importance_df.sort_values('Importance', ascending=False)

    return importance_df

# Получаем важность признаков для обеих моделей
feature_names = X.columns.tolist()
importance_simple = get_feature_importance(model_simple, feature_names)
importance_hidden = get_feature_importance(model_with_hidden, feature_names)

# Визуализация важности признаков для обеих моделей
plt.figure(figsize=(14, 10))

plt.subplot(2, 1, 1)
sns.barplot(x='Importance', y='Feature', data=importance_simple, palette='viridis')
plt.title('Важность признаков для простой модели')
plt.xlabel('Нормализованная важность')
plt.ylabel('Признак')
plt.tight_layout()

plt.subplot(2, 1, 2)
sns.barplot(x='Importance', y='Feature', data=importance_hidden, palette='viridis')
plt.title('Важность признаков для модели со скрытым слоем')
plt.xlabel('Нормализованная важность')
plt.ylabel('Признак')
plt.tight_layout()

plt.savefig('feature_importance.png')
plt.close()

# Вывод таблиц с важностью признаков
print("Важность признаков для простой модели:")
print(importance_simple[['Feature', 'Importance', 'Weight']])

print("\nВажность признаков для модели со скрытым слоем:")
print(importance_hidden[['Feature', 'Importance']])


<ipython-input-16-f061435bf2db>:43: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Importance', y='Feature', data=importance_simple, palette='viridis')
<ipython-input-16-f061435bf2db>:50: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Importance', y='Feature', data=importance_hidden, palette='viridis')


Важность признаков для простой модели:
                   Feature  Importance    Weight
0                   MedInc    0.256706  0.955952
6                 Latitude    0.236495 -0.880690
7                Longitude    0.220476 -0.821037
5                 AveOccup    0.053230 -0.198224
11          MedInc_Squared    0.051601 -0.192159
8         RoomBedroomRatio    0.043957 -0.163692
9        PopulationDensity    0.037777  0.140680
4               Population    0.025888 -0.096405
13  Income_Age_Interaction    0.023598  0.087876
3                AveBedrms    0.017830  0.066399
12        HouseAge_Squared    0.014338  0.053394
1                 HouseAge    0.008104  0.030179
2                 AveRooms    0.005608  0.020885
10      DistanceFromCenter    0.004391  0.016350

Важность признаков для модели со скрытым слоем:
                   Feature  Importance
6                 Latitude    0.147497
10      DistanceFromCenter    0.135941
7                Longitude    0.110451
0                   M

### Оценка

#### Оценка модели

In [17]:
# Оценка моделей на тестовом наборе данных
test_metrics_simple = evaluate_model(model_simple, test_loader)
test_metrics_hidden = evaluate_model(model_with_hidden, test_loader)

# Вывод метрик для обеих моделей на тестовых данных
print("Метрики на тестовых данных для простой модели:")
for metric, value in test_metrics_simple.items():
    print(f"{metric}: {value:.4f}")

print("\nМетрики на тестовых данных для модели со скрытым слоем:")
for metric, value in test_metrics_hidden.items():
    print(f"{metric}: {value:.4f}")

# Создание таблицы сравнения метрик
metrics_comparison = pd.DataFrame({
    'Метрика': list(test_metrics_simple.keys()),
    'Простая модель': list(test_metrics_simple.values()),
    'Модель со скрытым слоем': list(test_metrics_hidden.values())
})

print("\nСравнение метрик на тестовых данных:")
print(metrics_comparison)

# Визуализация сравнения метрик
plt.figure(figsize=(10, 6))
metrics_to_plot = ['MSE', 'RMSE', 'MAE']
comparison_data = metrics_comparison[metrics_comparison['Метрика'].isin(metrics_to_plot)]

comparison_data_melted = pd.melt(comparison_data,
                                 id_vars=['Метрика'],
                                 value_vars=['Простая модель', 'Модель со скрытым слоем'],
                                 var_name='Модель', value_name='Значение')

sns.barplot(x='Метрика', y='Значение', hue='Модель', data=comparison_data_melted)
plt.title('Сравнение метрик ошибок на тестовых данных')
plt.ylabel('Значение метрики')
plt.xlabel('Метрика')
plt.tight_layout()
plt.savefig('metrics_comparison.png')
plt.close()

# Визуализация R² отдельно (так как у него другой масштаб)
plt.figure(figsize=(8, 5))
r2_data = metrics_comparison[metrics_comparison['Метрика'] == 'R²']
r2_data_melted = pd.melt(r2_data,
                         id_vars=['Метрика'],
                         value_vars=['Простая модель', 'Модель со скрытым слоем'],
                         var_name='Модель', value_name='Значение')

sns.barplot(x='Модель', y='Значение', data=r2_data_melted)
plt.title('Сравнение коэффициента детерминации (R²) на тестовых данных')
plt.ylabel('R²')
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig('r2_comparison.png')
plt.close()


Метрики на тестовых данных для простой модели:
MSE: 0.4572
RMSE: 0.6762
MAE: 0.4969
R²: 0.6511

Метрики на тестовых данных для модели со скрытым слоем:
MSE: 0.2997
RMSE: 0.5475
MAE: 0.3762
R²: 0.7713

Сравнение метрик на тестовых данных:
  Метрика  Простая модель  Модель со скрытым слоем
0     MSE        0.457201                 0.299743
1    RMSE        0.676166                 0.547488
2     MAE        0.496918                 0.376158
3      R²        0.651101                 0.771260


#### Анализ ошибок

In [18]:
# Функция для получения предсказаний модели
def get_predictions(model, data_loader):
    model.eval()
    predictions = []
    actuals = []

    with torch.no_grad():
        for inputs, targets in data_loader:
            outputs = model(inputs)
            predictions.extend(outputs.numpy().flatten())
            actuals.extend(targets.numpy().flatten())

    return np.array(predictions), np.array(actuals)

# Получаем предсказания для обеих моделей
predictions_simple, actuals = get_predictions(model_simple, test_loader)
predictions_hidden, _ = get_predictions(model_with_hidden, test_loader)

# Создаем DataFrame с предсказаниями и фактическими значениями
predictions_df = pd.DataFrame({
    'Actual': actuals,
    'Predicted_Simple': predictions_simple,
    'Predicted_Hidden': predictions_hidden,
    'Error_Simple': actuals - predictions_simple,
    'Error_Hidden': actuals - predictions_hidden,
    'Abs_Error_Simple': np.abs(actuals - predictions_simple),
    'Abs_Error_Hidden': np.abs(actuals - predictions_hidden)
})

# Визуализация фактических и предсказанных значений
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.scatter(actuals, predictions_simple, alpha=0.5)
plt.plot([min(actuals), max(actuals)], [min(actuals), max(actuals)], 'r--')
plt.title('Фактические vs предсказанные значения (простая модель)')
plt.xlabel('Фактические значения')
plt.ylabel('Предсказанные значения')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.scatter(actuals, predictions_hidden, alpha=0.5)
plt.plot([min(actuals), max(actuals)], [min(actuals), max(actuals)], 'r--')
plt.title('Фактические vs предсказанные значения (модель со скрытым слоем)')
plt.xlabel('Фактические значения')
plt.ylabel('Предсказанные значения')
plt.grid(True)

plt.tight_layout()
plt.savefig('actual_vs_predicted.png')
plt.close()

# Визуализация распределения ошибок
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.histplot(predictions_df['Error_Simple'], kde=True)
plt.title('Распределение ошибок (простая модель)')
plt.xlabel('Ошибка')
plt.axvline(x=0, color='r', linestyle='--')
plt.grid(True)

plt.subplot(1, 2, 2)
sns.histplot(predictions_df['Error_Hidden'], kde=True)
plt.title('Распределение ошибок (модель со скрытым слоем)')
plt.xlabel('Ошибка')
plt.axvline(x=0, color='r', linestyle='--')
plt.grid(True)

plt.tight_layout()
plt.savefig('error_distribution.png')
plt.close()

# Визуализация ошибок в зависимости от фактических значений
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.scatter(actuals, predictions_df['Error_Simple'], alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.title('Ошибки vs фактические значения (простая модель)')
plt.xlabel('Фактические значения')
plt.ylabel('Ошибка')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.scatter(actuals, predictions_df['Error_Hidden'], alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.title('Ошибки vs фактические значения (модель со скрытым слоем)')
plt.xlabel('Фактические значения')
plt.ylabel('Ошибка')
plt.grid(True)

plt.tight_layout()
plt.savefig('errors_vs_actual.png')
plt.close()

# Анализ примеров с наибольшими ошибками
n_worst = 10
worst_predictions_simple = predictions_df.sort_values('Abs_Error_Simple', ascending=False).head(n_worst)
worst_predictions_hidden = predictions_df.sort_values('Abs_Error_Hidden', ascending=False).head(n_worst)

print(f"Топ-{n_worst} примеров с наибольшими ошибками (простая модель):")
print(worst_predictions_simple[['Actual', 'Predicted_Simple', 'Error_Simple']])

print(f"\nТоп-{n_worst} примеров с наибольшими ошибками (модель со скрытым слоем):")
print(worst_predictions_hidden[['Actual', 'Predicted_Hidden', 'Error_Hidden']])

# Статистика ошибок
error_stats = pd.DataFrame({
    'Статистика': ['Среднее', 'Медиана', 'Стандартное отклонение', 'Минимум', 'Максимум'],
    'Ошибка (простая модель)': [
        predictions_df['Error_Simple'].mean(),
        predictions_df['Error_Simple'].median(),
        predictions_df['Error_Simple'].std(),
        predictions_df['Error_Simple'].min(),
        predictions_df['Error_Simple'].max()
    ],
    'Абсолютная ошибка (простая модель)': [
        predictions_df['Abs_Error_Simple'].mean(),
        predictions_df['Abs_Error_Simple'].median(),
        predictions_df['Abs_Error_Simple'].std(),
        predictions_df['Abs_Error_Simple'].min(),
        predictions_df['Abs_Error_Simple'].max()
    ],
    'Ошибка (модель со скрытым слоем)': [
        predictions_df['Error_Hidden'].mean(),
        predictions_df['Error_Hidden'].median(),
        predictions_df['Error_Hidden'].std(),
        predictions_df['Error_Hidden'].min(),
        predictions_df['Error_Hidden'].max()
    ],
    'Абсолютная ошибка (модель со скрытым слоем)': [
        predictions_df['Abs_Error_Hidden'].mean(),
        predictions_df['Abs_Error_Hidden'].median(),
        predictions_df['Abs_Error_Hidden'].std(),
        predictions_df['Abs_Error_Hidden'].min(),
        predictions_df['Abs_Error_Hidden'].max()
    ]
})

print("\nСтатистика ошибок:")
print(error_stats)


Топ-10 примеров с наибольшими ошибками (простая модель):
       Actual  Predicted_Simple  Error_Simple
1649  5.00001          0.384331      4.615679
1865  5.00001          0.601493      4.398517
1140  5.00001          1.139383      3.860627
3710  4.50000          0.753651      3.746349
3693  5.00001          1.592859      3.407151
112   2.00000          5.380050     -3.380050
1138  5.00001          1.805690      3.194320
3761  5.00001          1.966418      3.033592
1250  0.67500          3.678408     -3.003408
872   5.00001          2.067486      2.932524

Топ-10 примеров с наибольшими ошибками (модель со скрытым слоем):
       Actual  Predicted_Hidden  Error_Hidden
1649  5.00001          0.439278      4.560732
2558  1.87500          5.588990     -3.713990
1140  5.00001          1.413445      3.586565
1865  5.00001          1.440434      3.559576
285   1.25000          4.598197     -3.348197
3710  4.50000          1.237940      3.262060
872   5.00001          1.944370      3.055640
42

Сравнение моделей

In [19]:
# Создадим сводную таблицу для сравнения моделей
comparison_table = pd.DataFrame({
    'Метрика': ['MSE', 'RMSE', 'MAE', 'R²', 'Время обучения (эпохи)', 'Количество параметров'],
    'Простая модель': [
        test_metrics_simple['MSE'],
        test_metrics_simple['RMSE'],
        test_metrics_simple['MAE'],
        test_metrics_simple['R²'],
        '100 эпох',
        sum(p.numel() for p in model_simple.parameters())
    ],
    'Модель со скрытым слоем': [
        test_metrics_hidden['MSE'],
        test_metrics_hidden['RMSE'],
        test_metrics_hidden['MAE'],
        test_metrics_hidden['R²'],
        '100 эпох',
        sum(p.numel() for p in model_with_hidden.parameters())
    ]
})

# Расчет процентного улучшения
improvement = pd.DataFrame({
    'Метрика': ['MSE', 'RMSE', 'MAE', 'R²'],
    'Улучшение (%)': [
        ((test_metrics_simple['MSE'] - test_metrics_hidden['MSE']) / test_metrics_simple['MSE']) * 100,
        ((test_metrics_simple['RMSE'] - test_metrics_hidden['RMSE']) / test_metrics_simple['RMSE']) * 100,
        ((test_metrics_simple['MAE'] - test_metrics_hidden['MAE']) / test_metrics_simple['MAE']) * 100,
        ((test_metrics_hidden['R²'] - test_metrics_simple['R²']) / test_metrics_simple['R²']) * 100
    ]
})

# Вывод таблицы сравнения
print("Сравнение моделей:")
print(comparison_table)

print("\nПроцентное улучшение метрик при использовании модели со скрытым слоем:")
print(improvement)

# Визуализация сравнения моделей
plt.figure(figsize=(10, 6))
metrics_to_plot = ['MSE', 'RMSE', 'MAE']
comparison_data = comparison_table[comparison_table['Метрика'].isin(metrics_to_plot)]

comparison_data_melted = pd.melt(comparison_data,
                                id_vars=['Метрика'],
                                value_vars=['Простая модель', 'Модель со скрытым слоем'],
                                var_name='Модель', value_name='Значение')

sns.barplot(x='Метрика', y='Значение', hue='Модель', data=comparison_data_melted)
plt.title('Сравнение метрик ошибок для моделей')
plt.ylabel('Значение метрики')
plt.xlabel('Метрика')
plt.tight_layout()
plt.savefig('models_comparison.png')
plt.close()

# Создадим график сравнения предсказаний моделей
plt.figure(figsize=(12, 8))

# Выберем случайные 100 примеров для наглядности
sample_indices = np.random.choice(len(predictions_df), 100, replace=False)
sample_df = predictions_df.iloc[sample_indices].sort_values('Actual')

plt.plot(range(len(sample_df)), sample_df['Actual'], 'o-', label='Фактические значения')
plt.plot(range(len(sample_df)), sample_df['Predicted_Simple'], 'o-', label='Простая модель')
plt.plot(range(len(sample_df)), sample_df['Predicted_Hidden'], 'o-', label='Модель со скрытым слоем')
plt.title('Сравнение предсказаний моделей на случайных примерах')
plt.xlabel('Индекс примера')
plt.ylabel('Цена жилья')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('predictions_comparison.png')
plt.close()

# Создадим диаграмму рассеяния для сравнения ошибок моделей
plt.figure(figsize=(10, 8))
plt.scatter(predictions_df['Abs_Error_Simple'], predictions_df['Abs_Error_Hidden'], alpha=0.5)
plt.plot([0, predictions_df['Abs_Error_Simple'].max()], [0, predictions_df['Abs_Error_Simple'].max()], 'r--')
plt.title('Сравнение абсолютных ошибок моделей')
plt.xlabel('Абсолютная ошибка простой модели')
plt.ylabel('Абсолютная ошибка модели со скрытым слоем')
plt.grid(True)
plt.tight_layout()
plt.savefig('errors_comparison.png')
plt.close()

# Подсчитаем, для скольких примеров модель со скрытым слоем дает лучший результат
better_hidden = (predictions_df['Abs_Error_Hidden'] < predictions_df['Abs_Error_Simple']).sum()
better_simple = (predictions_df['Abs_Error_Simple'] < predictions_df['Abs_Error_Hidden']).sum()
equal = (predictions_df['Abs_Error_Simple'] == predictions_df['Abs_Error_Hidden']).sum()

print(f"\nМодель со скрытым слоем дает лучший результат для {better_hidden} примеров ({better_hidden/len(predictions_df)*100:.2f}%)")
print(f"Простая модель дает лучший результат для {better_simple} примеров ({better_simple/len(predictions_df)*100:.2f}%)")
print(f"Обе модели дают одинаковый результат для {equal} примеров ({equal/len(predictions_df)*100:.2f}%)")


Сравнение моделей:
                  Метрика Простая модель Модель со скрытым слоем
0                     MSE       0.457201                0.299743
1                    RMSE       0.676166                0.547488
2                     MAE       0.496918                0.376158
3                      R²       0.651101                 0.77126
4  Время обучения (эпохи)       100 эпох                100 эпох
5   Количество параметров             15                     513

Процентное улучшение метрик при использовании модели со скрытым слоем:
  Метрика  Улучшение (%)
0     MSE      34.439437
1    RMSE      19.030523
2     MAE      24.301792
3      R²      18.454745

Модель со скрытым слоем дает лучший результат для 2644 примеров (64.05%)
Простая модель дает лучший результат для 1484 примеров (35.95%)
Обе модели дают одинаковый результат для 0 примеров (0.00%)
